# ESM-2 embeddings for IDR–GFP fusion constructs

Computes per-residue ESM-2 embeddings for the 3 126 fusion constructs in `constructs/all_constructs.fasta` and saves them to Google Drive.

**Model**: `facebook/esm2_t33_650M_UR50D` (1 280-dim, 650 M params). Lighter / heavier choices are documented in the config cell.

**Outputs** on Drive under `embeddings/`:

- `per_construct/<construct_id>.pt`  — per-residue embedding (float16) + IDR/full mean-pool + metadata
- `mean_pool_summary.parquet`        — single-row-per-construct table joining manifest metadata with mean-pooled vectors

**Long sequences (>1022 aa)** are handled with a sliding window (chunk = 1022, overlap = 200), averaging embeddings in overlap regions.

**Runtime**:

| GPU        | wallclock (3 126 seqs) |
|------------|------------------------|
| T4 (free)  | ~4–6 h                 |
| A100 (Pro) | ~1.5–2 h               |

The notebook is resume-safe — if the session disconnects, just re-run from the main loop and it skips already-completed constructs.

## Before running

1. Upload `constructs/all_constructs.fasta` to your Drive at `MyDrive/idrome_globular/constructs/all_constructs.fasta`.
2. In Colab: **Runtime → Change runtime type → GPU**.
3. Run the cells top-to-bottom.

## 1. Install deps and mount Drive

In [ ]:
!pip install -q transformers==4.41.0 biopython pyarrow tqdm

from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Edit the paths if you put files in a different Drive folder.

In [ ]:
import os

PROJECT_DIR       = '/content/drive/MyDrive/idrome_globular'
FASTA_PATH        = f'{PROJECT_DIR}/constructs/all_constructs.fasta'
OUT_DIR           = f'{PROJECT_DIR}/embeddings'
PER_CONSTRUCT_DIR = f'{OUT_DIR}/per_construct'
SUMMARY_PATH      = f'{OUT_DIR}/mean_pool_summary.parquet'

# Model choice. Trade-offs:
#   esm2_t12_35M_UR50D    480-dim   tiny, very fast
#   esm2_t30_150M_UR50D   640-dim   small, ~3x faster than 650M
#   esm2_t33_650M_UR50D   1280-dim  standard research model (default)
#   esm2_t36_3B_UR50D     2560-dim  heavy; needs A100 (FP16 ~7 GB)
MODEL_NAME        = 'facebook/esm2_t33_650M_UR50D'

MAX_TOKENS        = 1022          # 1024 - 2 (CLS, EOS)
CHUNK_OVERLAP     = 200           # for sliding window on long sequences
SAVE_FLOAT16      = True          # half-precision storage on disk
CHECKPOINT_EVERY  = 100           # flush mean-pool summary every N constructs

os.makedirs(PER_CONSTRUCT_DIR, exist_ok=True)
print('project    :', PROJECT_DIR)
print('fasta      :', FASTA_PATH)
print('out dir    :', OUT_DIR)
print('model      :', MODEL_NAME)

## 3. Load FASTA and parse metadata from headers

Each header looks like:
```
>Q86XK2_896_927__GFPp15__Ntail | idr=Q86XK2_896_927 | gfp=GFP+15 | topology=Ntail | N_idr=32 | ...
```

We use the first token (`construct_id`) as the unique key and split the `| key=value` pairs to keep the metadata.

In [ ]:
from Bio import SeqIO

records = []
for rec in SeqIO.parse(FASTA_PATH, 'fasta'):
    meta = {}
    if rec.description and '|' in rec.description:
        for piece in rec.description.split('|')[1:]:
            piece = piece.strip()
            if '=' in piece:
                k, v = piece.split('=', 1)
                meta[k.strip()] = v.strip()
    records.append({'construct_id': rec.id, 'sequence': str(rec.seq), **meta})

lengths = [len(r['sequence']) for r in records]
n_long  = sum(1 for L in lengths if L > MAX_TOKENS)
print(f'loaded {len(records)} constructs')
print(f'length range : {min(lengths)} - {max(lengths)}')
print(f'long (>{MAX_TOKENS} aa) : {n_long} (sliding window will be used)')

## 4. Load ESM-2

In [ ]:
import torch
from transformers import EsmTokenizer, EsmModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device :', DEVICE)
if DEVICE == 'cuda':
    print('GPU    :', torch.cuda.get_device_name())

tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
if DEVICE == 'cuda':
    model = model.half()         # FP16 for speed / memory

HIDDEN_DIM = model.config.hidden_size
print('hidden dim :', HIDDEN_DIM)

## 5. Embedding function (single-sequence, sliding window for long seqs)

In [ ]:
@torch.inference_mode()
def embed_sequence(seq: str,
                   max_len: int = MAX_TOKENS,
                   overlap: int = CHUNK_OVERLAP) -> torch.Tensor:
    """Return per-residue embedding (L, D) on CPU as float32."""
    L = len(seq)
    D = HIDDEN_DIM

    if L <= max_len:
        toks = tokenizer(seq, return_tensors='pt').to(DEVICE)
        out  = model(**toks).last_hidden_state          # (1, L+2, D)
        return out[0, 1:-1].float().cpu()               # strip CLS/EOS -> (L, D)

    accum  = torch.zeros(L, D, dtype=torch.float32)
    weight = torch.zeros(L,    dtype=torch.float32)
    step   = max_len - overlap
    starts = list(range(0, L, step))
    if starts[-1] + max_len < L:
        starts.append(L - max_len)                       # ensure last window reaches L
    for start in starts:
        end = min(start + max_len, L)
        chunk = seq[start:end]
        toks  = tokenizer(chunk, return_tensors='pt').to(DEVICE)
        out   = model(**toks).last_hidden_state
        emb   = out[0, 1:-1].float().cpu()
        accum[start:end]  += emb
        weight[start:end] += 1.0
    return accum / weight.unsqueeze(1)

## 6. Main loop — compute & save embeddings

Resume-safe: skips any construct that already has a `.pt` file. Periodically flushes the mean-pool summary to Drive.

In [ ]:
import time
import pandas as pd
from tqdm.auto import tqdm

META_KEYS = ['idr', 'gfp', 'topology', 'N_idr', 'N_construct',
             'ncpr_idr', 'fcr_idr', 'kappa_idr', 'nu_idr_free', 'Z_gfp']

def _maybe_num(s: str):
    try:    return float(s)
    except: return s

summary_buffer: list[dict] = []

def flush_summary():
    if not summary_buffer:
        return
    df = pd.DataFrame(summary_buffer)
    if os.path.exists(SUMMARY_PATH):
        old = pd.read_parquet(SUMMARY_PATH)
        df = (pd.concat([old, df], ignore_index=True)
                .drop_duplicates('construct_id', keep='last'))
    df.to_parquet(SUMMARY_PATH, index=False)
    summary_buffer.clear()

# sort short-first so any OOM on very long seqs surfaces only after we have
# the bulk of the data already saved
records_sorted = sorted(records, key=lambda r: len(r['sequence']))
t0 = time.time()
n_skipped = 0

for rec in tqdm(records_sorted):
    cid = rec['construct_id']
    out_path = os.path.join(PER_CONSTRUCT_DIR, f'{cid}.pt')
    if os.path.exists(out_path):
        n_skipped += 1
        continue

    emb = embed_sequence(rec['sequence'])             # (L, D) float32 CPU

    topo  = rec.get('topology', '')
    n_idr = int(float(rec.get('N_idr', 0)))
    if topo == 'Ntail':
        idr_emb = emb[:n_idr]
    elif topo == 'Ctail':
        idr_emb = emb[-n_idr:]
    else:
        idr_emb = emb

    mean_full = emb.mean(dim=0)
    mean_idr  = idr_emb.mean(dim=0)

    payload = {
        'construct_id'  : cid,
        'sequence'      : rec['sequence'],
        'residue_emb'   : emb.to(torch.float16) if SAVE_FLOAT16 else emb,
        'mean_pool_full': mean_full,
        'mean_pool_idr' : mean_idr,
        'metadata'      : {k: _maybe_num(rec.get(k, '')) for k in META_KEYS},
    }
    torch.save(payload, out_path)

    row = {'construct_id': cid}
    for k in META_KEYS:
        row[k] = _maybe_num(rec.get(k, ''))
    row.update({f'mean_full_{i}': float(mean_full[i]) for i in range(HIDDEN_DIM)})
    row.update({f'mean_idr_{i}' : float(mean_idr[i])  for i in range(HIDDEN_DIM)})
    summary_buffer.append(row)

    if len(summary_buffer) >= CHECKPOINT_EVERY:
        flush_summary()

flush_summary()
print(f'\ndone in {(time.time()-t0)/60:.1f} min')
print(f'  newly embedded : {len(records_sorted) - n_skipped}')
print(f'  skipped (cached): {n_skipped}')

## 7. Sanity check

In [ ]:
import pandas as pd, torch, os

summary = pd.read_parquet(SUMMARY_PATH)
print('summary shape   :', summary.shape)
print('unique constructs:', summary['construct_id'].nunique())
print('metadata cols   :', [c for c in summary.columns if not c.startswith('mean_')][:12])

cid = summary['construct_id'].iloc[0]
p = torch.load(os.path.join(PER_CONSTRUCT_DIR, f'{cid}.pt'), map_location='cpu')
print(f'\n--- {cid} ---')
for k, v in p.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:18}  shape={tuple(v.shape)}  dtype={v.dtype}')
    elif isinstance(v, dict):
        print(f'  {k:18}  {v}')
    else:
        print(f'  {k:18}  {type(v).__name__} (len={len(v) if hasattr(v, "__len__") else "-"})')

## What you can do with the outputs

Once the loop finishes, you have two complementary representations:

1. **`mean_pool_summary.parquet`** — one row per construct, 2 560 numeric columns (`mean_full_*` and `mean_idr_*`) plus all manifest metadata. Load it directly into pandas for clustering, UMAP, regression on ΔRg, etc.:
   ```python
   import pandas as pd
   df = pd.read_parquet(SUMMARY_PATH)
   X = df[[c for c in df.columns if c.startswith('mean_idr_')]].to_numpy()
   ```

2. **`per_construct/<id>.pt`** — full per-residue embeddings (float16). Use when you need residue-resolved signals, e.g. tracking the embedding of IDR residues nearest the GFP fusion point versus distal ones.

Typical next analyses:

- **UMAP / PCA on `mean_idr_*`** — cluster IDRs by ESM-2 representation, see whether the clusters align with `NCPR`, `kappa`, IDRome class.
- **Joint embedding → ΔRg regression**: once CALVADOS results land, fit an MLP that maps `(mean_idr_*, Z_gfp, topology)` → `ΔRg`. Compare against the analytical regression on `(NCPR, kappa, FCR, length, Z_gfp)` to see if ESM captures features the hand-engineered descriptors miss.
- **Construct-pair contrasts**: same IDR, different GFP. The difference between two mean-pooled embeddings isolates the *context effect* of the GFP variant on the IDR representation.